In [0]:
!pip install azure-search-documents
!pip install azure-identity
!pip install openai
!pip install --upgrade typing_extensions
dbutils.library.restartPython()

In [0]:
index_name='index-q2poc-abstract-summary'

#Azure AI Searchの情報を設定
service_name = 'ai-search-tmc-allgenai-poc'
api_version = "2024-05-01-preview"
search_api_key = 'xQ07KHXJ2Kms79xiw434BUWcSvik4nxLW3qlHg9IXuAzSeA31ZoX'
search_endpoint = "https://ai-search-tmc-allgenai-poc.search.windows.net"

openai_endpoint = "https://openai-tmcallgenai-japan.openai.azure.com/"
openai_apikey = "3d192023257f481693ccf085076fea6f"

excel_path = '/Workspace/Users/yoichiro.abe@avanade.com/マテQ回答結果_抽象.xlsx'

resutl_file = "abstract_summary_results"

qa_file_path = "/Workspace/Users/shaun.baron.furuyama@avanade.com/抽象質問_20240913.xlsx"
# "/Workspace/Users/yoichiro.abe@avanade.com/マテQ回答生成ログ抽象用.xlsx"

In [0]:
# ハイブリッド検索
def hybridSearch(query):
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient
    from azure.search.documents.indexes import SearchIndexClient
    from azure.search.documents.models import VectorizedQuery

    def get_embeddings(text: list[str]):
        # There are a few ways to get embeddings. This is just one example.
        import openai
        print(f"search_endpoint: {search_endpoint} api_key: {search_api_key} api_version: {api_version}")
        client = openai.AzureOpenAI(
            azure_endpoint=openai_endpoint,
            api_key=openai_apikey,
            api_version="2024-06-01"
        )
        embedding = client.embeddings.create(input=[text], model="text-embedding-3-large")
        # print(f"embedding.data={embedding.data[0].embedding}")
        return embedding.data[0].embedding

    credential = AzureKeyCredential(search_api_key)
    embd = get_embeddings(query) 
    # print(f"resp={resp}")
    vector_query2 = VectorizedQuery(vector=embd, k_nearest_neighbors=3, fields="SummaryVector")

    search_client = SearchClient(endpoint=search_endpoint,
                        index_name=index_name,
                        credential=credential)
    results =  search_client.search(query_type='semantic',
        search_text=query, 
        vector_queries=[vector_query2],
        semantic_configuration_name="q2-semantic_configuration",
        # include_total_count=True # 近似値だしパフォーマンス影響あるらしいので一旦削除
        top=10
        )

    import json
    rag_list = list()
    for result in results:
        # print(f"result:{result['Title']} score:{result['@search.score']}rerank:{result['@search.reranker_score']}")
        text = json.dumps(result, ensure_ascii=False)
        rag_list.append(text)

    # print(f"marged_result={marged_result}")
    # return marged_result
    return rag_list

In [0]:
# 回答生成関数

import openai
client = openai.AzureOpenAI(
    azure_endpoint=openai_endpoint,
    api_key=openai_apikey,
    api_version="2024-04-01-preview"
)
# 質問に対する回答を生成する関数
def get_answer(question,system_prompt):
    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        # messages= [{"role": "user", "content": question}]
        messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ],
    )
    return response.choices[0].message.content.strip()


In [0]:
# 回答生成
system_prompt = """あなたは材料学のプロフェッショナルで、自動車メーカーで働いています。設計者からの質問に対して材料学の見地から、回答やアドバイスをしてください。
"""

def createPrompt(query, hybrid_search_result):
    prompt = f"""以下の質問文に対し、データソースを基にして適切に回答してください。精緻さよりも分かりやすさを心がけてください。回答に使用した文章がデータソース内にあれば、使用したデータソースのみ、そのTitleを最後に記載してください。\n\n### 質問文\n
    {query}
    \n\n### データソース\n
    {hybrid_search_result}
    """
    return prompt



# # 質問を入力して回答を取得
# answer = get_answer(prompt,system_prompt)
# # print("質問:", prompt)
# print(answer)

# 回答生成を連続で実施し、データベースに保存する

In [0]:
import pandas as pd
import json

# qa_df = pd.read_excel(qa_file_path,sheet_name="0912") 
qa_df = pd.read_excel(qa_file_path) 

qa_results = []

for index, row in qa_df.iterrows():
    query = row['質問']
    print(f"process start:{query}")
    # human_answer = row['回答']
    comment = row['コメント']
    human_answer = ""
    rag_list = hybridSearch(query)
    prompt = createPrompt(query, rag_list)
    
    documenttitle = ""
    for src in rag_list:
        item = json.loads(src)
        ragitem = f"""Title:{item['Title']} MajorCategory:{item['MajorCategory']} SubCategory:{item['SubCategory']}\n"""
        documenttitle += ragitem

    for i in range(1):
        rag_answer = get_answer(prompt, system_prompt)
        qa_results.append({
            '質問ナンバー' : index + 1,
            '質問' : query,
            'RAG回答' : rag_answer,
            # '人間回答' : human_answer,
            'コメント': comment,
            '検索されたドキュメント':documenttitle
        })

qa_results_df = pd.DataFrame(qa_results)

display(qa_results_df)

In [0]:
# Save the DataFrame as a table in the specified schema
from datetime import datetime,timedelta

# 現在の日時を取得
now = datetime.now()
# yyyyMMddHHmm形式の文字列に変換
new_time = now + timedelta(hours=9)
formatted_date = new_time.strftime('%Y%m%d_%H%M')

In [0]:
qa_results_df.to_excel(excel_path, sheet_name=f'{formatted_date}') 

In [0]:
from datetime import datetime, timedelta

def save_df_to_catalog(df, table_name: str):
    # 現在のUTC日時を取得
    now_utc = datetime.utcnow()
    # 日本時間 (UTC+9) に変換
    now_japan = now_utc + timedelta(hours=9)
    # yyyyMMddHHmm形式の文字列に変換
    formatted_date_japan = now_japan.strftime('%Y%m%d_%H%M')

    df = spark.createDataFrame(df)

    # Create the schema in the specified catalog and schema
    spark.sql("CREATE SCHEMA IF NOT EXISTS hive_metastore.poc2")

    # Save the DataFrame as a table in the specified schema
    df.write.mode("overwrite").saveAsTable(f"poc2.{table_name}_{formatted_date_japan}")
    
    print(f"Table 'poc2.{table_name}_{formatted_date_japan}' was created.")
    check_df = spark.table(f"poc2.{table_name}_{formatted_date_japan}")
    display(check_df)

save_df_to_catalog(qa_results_df, "abstract_qa_results")

In [0]:
# from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
# from datetime import datetime, timedelta

# # 現在のUTC日時を取得
# now_utc = datetime.utcnow()
# # 日本時間 (UTC+9) に変換
# now_japan = now_utc + timedelta(hours=9)
# # yyyyMMddHHmm形式の文字列に変換
# formatted_date_japan = now_japan.strftime('%Y%m%d_%H%M')
# display(formatted_date_japan)

# # Define the schema
# schema = StructType([
#     StructField("質問ナンバー", IntegerType(), True),
#     StructField("質問", StringType(), True),
#     StructField("RAG回答", StringType(), True),
#     StructField("人間回答", StringType(), True),
#     StructField("検索されたドキュメント", StringType(), True)
# ])

# # Create the schema in the specified catalog and schema
# spark.sql("CREATE SCHEMA IF NOT EXISTS hive_metastore.poc2")

# # Save the DataFrame as a table in the specified schema
# from datetime import datetime,timedelta


# spark.createDataFrame(qa_results_df, schema=schema).write.mode("overwrite").saveAsTable(f"poc2.{resutl_file}_{formatted_date_japan}")


In [0]:
# import pandas as pd

# qa_file_path = '/dbfs/FileStore/マテQ回答生成ログ.xlsx' #TODO: swithc this to the appropriate sheet
# qa_df = pd.read_excel(qa_file_path,sheet_name="20240906") 
# qa_df = qa_df.head(4)

# display(qa_df)